# 📄 NLP Contract Analysis Demo: Structured AI-Powered Legal Document Understanding

---

## 🎯 Overall Purpose of This Notebook

This notebook demonstrates a **complete, end-to-end Natural Language Processing (NLP) workflow** for contract analysis using a large language model.  

In many real business settings, contracts are long, dense, repetitive, and expensive to review manually.  
A modern NLP pipeline can help analysts, accountants, managers, auditors, and legal teams to:

- identify the **parties** involved in an agreement,
- summarize the **main purpose** of the contract,
- extract **key business terms** such as dates, payment terms, and termination conditions,
- identify **major risks** and legal red flags,
- generate a **decision-oriented executive summary**.

This notebook uses the **Qwen model family** through a hard-coded API configuration, following the same practical demonstration style as the OCR notebook.

---

## 🧠 Why This Notebook Is Structured This Way

Rather than jumping directly into one large block of code, the notebook is divided into a sequence of well-defined stages:
1. environment setup,
2. import and configuration,
3. dataset download and loading,
4. contract preview,
5. LLM-based contract analysis,
6. final reporting and structured output display.

---

## 📥 Global Input and Output of the Entire Notebook

### 📥 Notebook Inputs

This notebook takes the following inputs:

1. **A contract dataset**
   - We load contract text from the **CUAD** dataset in a Colab-friendly way.
   - The notebook prepares a small fixed set of sample contracts for demonstration.

2. **A hard-coded Qwen API configuration**
   - API key
   - endpoint URL
   - HTTP request headers

3. **Prompt instructions**
   - These tell the model exactly how to analyze each contract.

---

### 📤 Notebook Outputs

This notebook produces the following outputs:

1. **Clean previews of the original contract text**
2. **Structured AI analysis for each contract**
3. **A document-level summary table**
4. **A full analysis report table**
5. **A classroom-friendly final narrative report**

---

## 🔄 End-to-End Notebook Flow

```text
┌──────────────────────────────┐
│      Contract Dataset        │
│      (CUAD text files)       │
└──────────────┬───────────────┘
               │
               ▼
┌──────────────────────────────┐
│  Fixed Demo Contract Loader  │
│ (Colab-friendly data access) │
└──────────────┬───────────────┘
               │
               ▼
┌──────────────────────────────┐
│    Original Contract Text    │
│        Preview Stage         │
└──────────────┬───────────────┘
               │
               ▼
┌──────────────────────────────┐
│   Qwen NLP Analysis Engine   │
│  (Prompted contract reader)  │
└──────────────┬───────────────┘
               │
               ▼
┌──────────────────────────────┐
│  Structured Analysis Output  │
│  - parties                   │
│  - purpose                   │
│  - obligations               │
│  - key terms                 │
│  - risk assessment           │
│  - executive summary         │
└──────────────┬───────────────┘
               │
               ▼
┌──────────────────────────────┐
│   Final Report and Tables    │
└──────────────────────────────┘
```

---

## 1. Environment Setup

This section prepares the notebook runtime.

### What this section does
This step installs the Python packages needed for:
- dataset loading,
- API requests,
- tabular summaries,
- notebook display.

### Why this step matters
A Colab runtime starts from a relatively clean environment.  
If we do not explicitly install the required libraries, later cells may fail because the dependencies are missing.

### Conceptual mini-flow

```text
Fresh Colab Runtime
        ↓
Install Required Libraries
        ↓
Ready-to-run NLP Environment
```

In [12]:

# Install the libraries required for this notebook.
# Each package supports a different part of the end-to-end workflow.

# Install the Hugging Face datasets package so we can load CUAD in a Colab-friendly way.
!pip -q install datasets

# Install pandas for building final summary tables.
!pip -q install pandas

# Install requests for direct HTTP calls to the Qwen API endpoint.
!pip -q install requests

## 2. Import Libraries and Configure the API

This section performs the core notebook configuration.

### What this section does
This step:
- imports all Python libraries used later in the notebook,
- defines the hard-coded API key and endpoint,
- creates reusable helper functions for printing sections and making API requests.

### Why this step matters
The notebook needs a single, centralized configuration point so the rest of the workflow stays clean and readable.  
Putting all imports, constants, and helper utilities in one place makes the notebook easier to maintain and easier for students to understand.

### Conceptual mini-flow

```text
Installed Libraries
        ↓
Import Modules
        ↓
Define API Configuration
        ↓
Build Reusable Utilities
        ↓
Notebook Ready for Analysis
```

In [13]:

# Import the requests library so the notebook can send HTTP requests to the Qwen API.
import requests

# Import json so we can safely work with structured API responses.
import json

# Import os so we can inspect paths and environment-related information when needed.
import os

# Import re for regular-expression-based text cleaning and parsing.
import re

# Import time so we can add small pauses between API calls if needed.
import time

# Import pandas so we can build structured summary tables at the end of the notebook.
import pandas as pd

# Import datetime so timestamps can be added to outputs if desired.
from datetime import datetime

# Import display from IPython so DataFrames can be rendered neatly inside the notebook.
from IPython.display import display

# Import load_dataset from the datasets library so CUAD can be downloaded directly in Colab.
from datasets import load_dataset

# Define the hard-coded API key exactly as used in the original notebook style.
API_KEY = "sk-cJvvaaDosK2UQKWvPeyK7HyrRK004ul6L9IN7YdZm00cd7xT"

# Define the complete DashScope-compatible chat completions endpoint.
BASE_URL = "https://api.bianxie.ai/v1/chat/completions"

# Define the default model used for contract analysis.
DEFAULT_MODEL_NAME = "qwen-plus"

# Define a helper function that prints a visible section header for cleaner notebook output.
def print_section(title):
    # Print a long separator line.
    print("\n" + "=" * 100)
    # Print the section title itself.
    print(title)
    # Print another separator line.
    print("=" * 100)

# Define a helper function that creates the HTTP headers required by the API.
def setup_qwen_client():
    # Build the authorization and content-type headers in dictionary form.
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }
    # Return the completed header dictionary.
    return headers

# Create the reusable headers object once so later API calls remain clean.
HEADERS = setup_qwen_client()

# Print a short configuration summary so the user can confirm that setup succeeded.
print_section("STEP 1 — API CONFIGURATION")
print(f"✅ API client configured")
print(f"🔑 API key prefix: {API_KEY[:10]}...{API_KEY[-4:]}")
print(f"🌐 API endpoint: {BASE_URL}")
print(f"🧠 Default model: {DEFAULT_MODEL_NAME}")


STEP 1 — API CONFIGURATION
✅ API client configured
🔑 API key prefix: sk-cJvvaaD...d7xT
🌐 API endpoint: https://api.bianxie.ai/v1/chat/completions
🧠 Default model: qwen-plus


## 3. Test the API Connection and Load the Contract Dataset

This section combines two closely related setup tasks:
1. verify that the LLM connection works,
2. load a reproducible set of demo contracts.

### What this section does
This step:
- sends a short test request to the API,
- downloads the CUAD dataset in a Colab-friendly format,
- selects a small fixed set of contract examples for the notebook.

### Why this step matters
Before running expensive or time-consuming analysis, we should confirm two things:
- the model can actually be reached,
- the notebook has valid contract text to analyze.

This prevents downstream errors and makes the workflow more robust.

### Conceptual mini-flow

```text
API Configuration + Colab Runtime
              ↓
      Test API Connection
              ↓
      Load CUAD Dataset
              ↓
 Select Fixed Demo Contracts
              ↓
Ready for Contract Preview
```

In [14]:

# Define a helper function that sends a very small test request to the API.
def test_api_connection():
    # Print a visible step header.
    print_section("STEP 2 — API CONNECTION TEST")

    # Build a minimal payload for a lightweight connection test.
    test_payload = {
        "model": DEFAULT_MODEL_NAME,
        "messages": [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "Reply with exactly: API connection successful."}
        ],
        "max_tokens": 20,
        "temperature": 0.0
    }

    # Try to call the API and validate the response.
    try:
        # Send the POST request to the configured endpoint.
        response = requests.post(BASE_URL, headers=HEADERS, json=test_payload, timeout=60)

        # Raise an exception immediately if the HTTP status is not successful.
        response.raise_for_status()

        # Convert the response body from JSON text into a Python dictionary.
        response_json = response.json()

        # Extract the assistant content from the standard chat-completions response structure.
        assistant_text = response_json["choices"][0]["message"]["content"]

        # Print a success message and the model reply.
        print("✅ API test succeeded")
        print(f"🤖 Test response: {assistant_text}")

        # Return True so later cells know the API is working.
        return True

    # Catch any exception and report it clearly.
    except Exception as e:
        # Print a failure message.
        print(f"❌ API test failed: {e}")

        # Return False so later cells can avoid analysis requests.
        return False

# Run the API connectivity test and store the result for downstream logic.
api_working = test_api_connection()


STEP 2 — API CONNECTION TEST
✅ API test succeeded
🤖 Test response: API connection successful.


In [15]:
# Define a helper function that loads CUAD contracts from GitHub instead of local folder.
def load_cuad_demo_contracts():
    # Print a visible step header to make notebook output structured and readable.
    print_section("STEP 3 — LOAD CONTRACTS FROM GITHUB")

    # Define the GitHub repository API URL to list files in the repo.
    api_url = "https://api.github.com/repos/MaggieChen7/CUAD/contents"

    # Send a GET request to GitHub API to retrieve file metadata.
    response = requests.get(api_url)

    # Convert the JSON response into Python data structure (list of files).
    files = response.json()

    # Filter only .txt files (contracts).
    txt_files = [f for f in files if f["name"].endswith(".txt")]

    # Sort file list to ensure deterministic selection (important for teaching reproducibility).
    txt_files = sorted(txt_files, key=lambda x: x["name"])

    # Check if there are enough files available.
    if len(txt_files) == 0:
        raise ValueError("❌ No .txt contract files found in the GitHub repo.")

    # Define how many contracts we want to load for the demo.
    num_samples = 3  # Keep it small for readability and performance.

    # Select the first 3 files (deterministic instead of random).
    selected_files = txt_files[:num_samples]

    # Create an empty list to store structured contract data.
    selected_contracts = []

    # Loop through selected contract files one by one.
    for index, file_info in enumerate(selected_files, start=1):

        # Extract filename from GitHub metadata.
        filename = file_info["name"]

        # Get the raw download URL for the file.
        download_url = file_info["download_url"]

        # Send request to download the file content.
        file_response = requests.get(download_url)

        # Decode file content into text format.
        contract_text = file_response.text

        # Skip files that are too short or empty.
        if not contract_text or len(contract_text.strip()) < 1500:
            continue

        # Create a readable display title (remove ".txt" suffix).
        display_title = filename.replace(".txt", "")

        # Append structured contract information to the list.
        selected_contracts.append({
            "sample_id": f"contract_{index}",  # Unique ID for each contract.
            "filename": filename,  # Original file name.
            "display_title": display_title,  # Clean title for display.
            "content": contract_text,  # Full contract text.
            "word_count": len(contract_text.split())  # Word count for analysis.
        })

    # Check if we successfully loaded any valid contracts.
    if not selected_contracts:
        raise ValueError("❌ No suitable contracts found after filtering.")

    # Print a summary of loaded contracts.
    print(f"✅ Loaded {len(selected_contracts)} fixed demo contracts")

    # Loop through loaded contracts and print basic info.
    for contract in selected_contracts:
        print(f"📄 {contract['display_title']} | {contract['word_count']} words")

    # Return the structured contract list.
    return selected_contracts


# Call the function and store results in the global variable `contracts`.
contracts = load_cuad_demo_contracts()


STEP 3 — LOAD CONTRACTS FROM GITHUB
✅ Loaded 3 fixed demo contracts
📄 2ThemartComInc_19990826_10-12G_EX-10.10_6700288_EX-10.10_Co-Branding Agreement_ Agency Agreement | 4467 words
📄 ABILITYINC_06_15_2020-EX-4.25-SERVICES AGREEMENT | 4115 words
📄 ACCELERATEDTECHNOLOGIESHOLDINGCORP_04_24_2003-EX-10.13-JOINT VENTURE AGREEMENT | 1911 words


## 4. Preview the Original Contract Text

This section presents the raw source documents before analysis.

### What this section does
This step prints a readable preview of each selected contract.

### Why this step matters
We should see the original legal text before seeing the model output.  
That makes the notebook much more interpretable: users can compare what the model says with what the source document actually contains.

### Conceptual mini-flow

```text
Selected Contract Records
          ↓
Read Original Text
          ↓
Display Structured Preview
          ↓
Human Understanding Before AI Analysis
```

In [16]:

# Print a visible section title for the preview stage.
print_section("STEP 4 — DISPLAY ORIGINAL CONTRACT TEXT")

# Loop through the selected contracts one by one.
for index, contract in enumerate(contracts, start=1):
    # Print a contract-level separator.
    print("\n" + "-" * 100)

    # Print the contract number and display title.
    print(f"📄 CONTRACT {index}: {contract['display_title']}")

    # Print the file-style name used in the notebook record.
    print(f"📁 File name: {contract['filename']}")

    # Print the approximate word count.
    print(f"🧮 Word count: {contract['word_count']}")

    # Print a preview label.
    print("\n📝 ORIGINAL TEXT PREVIEW")
    print("-" * 60)

    # Create a truncated preview so notebook output stays readable.
    preview_text = contract["content"][:6000]

    # Add a continuation marker when the contract is longer than the preview window.
    if len(contract["content"]) > 6000:
        preview_text += "\n\n... [TEXT TRUNCATED FOR PREVIEW]"

    # Print the preview text itself.
    print(preview_text)


STEP 4 — DISPLAY ORIGINAL CONTRACT TEXT

----------------------------------------------------------------------------------------------------
📄 CONTRACT 1: 2ThemartComInc_19990826_10-12G_EX-10.10_6700288_EX-10.10_Co-Branding Agreement_ Agency Agreement
📁 File name: 2ThemartComInc_19990826_10-12G_EX-10.10_6700288_EX-10.10_Co-Branding Agreement_ Agency Agreement.txt
🧮 Word count: 4467

📝 ORIGINAL TEXT PREVIEW
------------------------------------------------------------
CO-BRANDING AND ADVERTISING AGREEMENT

THIS CO-BRANDING AND ADVERTISING AGREEMENT (the "Agreement") is made as of June 21, 1999 (the "Effective Date") by and between I-ESCROW, INC., with its principal place of business at 1730 S. Amphlett Blvd., Suite 233, San Mateo, California 94402 ("i-Escrow"), and 2THEMART.COM, INC. having its principal place of business at 18301 Von Karman Avenue, 7th Floor, Irvine, California 92612 ("2TheMart").

1. DEFINITIONS.

(a) "CONTENT" means all content or information, in any medium, provide

## 5. Define the NLP Analysis Pipeline and Run Contract Analysis

This section contains the analytical core of the notebook.

### What this section does
This step:
- defines the prompt used for contract review,
- sends each contract to the Qwen model,
- stores the structured analysis results.

### Why this step matters
This is the stage where raw contract text is transformed into usable business intelligence.  
The prompt is intentionally structured so the model returns:
- parties,
- purpose,
- obligations,
- key terms,
- risk assessment,
- an executive summary.

### Conceptual mini-flow

```text
Original Contract Text
          ↓
Structured Analysis Prompt
          ↓
Qwen Contract Reading
          ↓
Structured Contract Analysis
```

In [17]:

# Define a helper function that builds the analysis prompt for one contract.
def build_contract_analysis_prompt(contract_text):
    # Return a structured prompt that asks the model to analyze the contract clearly and consistently.
    return f"""
You are an expert contract analyst supporting accounting, audit, legal, and business review workflows.

Please analyze the following contract and produce a structured report using the exact section headings below.

CONTRACT TO ANALYZE:
--------------------
{contract_text}

REQUIRED OUTPUT FORMAT:

PARTIES:
- Party A:
- Party B:

MAIN PURPOSE:
- Briefly explain what this contract is about in 2-4 sentences.

MAIN RIGHTS AND OBLIGATIONS:
- Party A obligations:
- Party B obligations:

KEY TERMS:
- Payment or value terms:
- Duration or important dates:
- Termination conditions:
- Liability or indemnity observations:
- Governing law or jurisdiction, if available:

RISK LEVEL:
- Overall risk level: Low / Medium / High
- Main risk factors:

EXECUTIVE SUMMARY:
- Write a concise executive summary in 4-6 bullet points for a business audience.
""".strip()

# Define a helper function that calls the Qwen API for one contract.
def analyze_contract(contract_text):
    # Build the full prompt for the current contract.
    prompt = build_contract_analysis_prompt(contract_text)

    # Create the chat-completions payload sent to the API.
    payload = {
        "model": DEFAULT_MODEL_NAME,
        "messages": [
            {
                "role": "system",
                "content": "You are a precise, business-oriented legal contract analysis assistant."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        "max_tokens": 1500,
        "temperature": 0.1
    }

    # Send the request to the API inside a try/except block for safer error handling.
    try:
        # Submit the POST request.
        response = requests.post(BASE_URL, headers=HEADERS, json=payload, timeout=120)

        # Raise an exception if the response status is not successful.
        response.raise_for_status()

        # Parse the JSON response body.
        response_json = response.json()

        # Extract the assistant's analysis text.
        analysis_text = response_json["choices"][0]["message"]["content"]

        # Return the model output.
        return analysis_text

    # Catch any exception and return a readable error marker instead of crashing the entire notebook.
    except Exception as e:
        # Return a formatted error string so the failure is visible in downstream reports.
        return f"ERROR: API request failed — {e}"

# Print a visible title before running the contract analysis stage.
print_section("STEP 5 — RUN ANALYSIS ON ALL CONTRACTS")

# Create an empty list that will store the final structured analysis records.
analysis_results = []

# Loop through the selected contracts one by one.
for contract in contracts:
    # Print a visible per-contract progress message.
    print(f"🔍 Analyzing: {contract['display_title']}")

    # Run the model analysis only if the API connection test succeeded earlier.
    if api_working:
        # Generate the analysis text for the current contract.
        analysis_text = analyze_contract(contract["content"])
    else:
        # Create a fallback message if the API is unavailable.
        analysis_text = "ERROR: Analysis skipped because the API connection test failed."

    # Build a structured result object that combines input metadata and output text.
    analysis_results.append({
        "sample_id": contract["sample_id"],
        "title": contract["display_title"],
        "filename": contract["filename"],
        "word_count": contract["word_count"],
        "analysis_text": analysis_text,
        "analysis_timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    })

    # Pause briefly between requests to keep the workflow polite and stable.
    time.sleep(1)

# Print a completion message after all contracts have been processed.
print("✅ Contract analysis stage completed")


STEP 5 — RUN ANALYSIS ON ALL CONTRACTS
🔍 Analyzing: 2ThemartComInc_19990826_10-12G_EX-10.10_6700288_EX-10.10_Co-Branding Agreement_ Agency Agreement
🔍 Analyzing: ABILITYINC_06_15_2020-EX-4.25-SERVICES AGREEMENT
🔍 Analyzing: ACCELERATEDTECHNOLOGIESHOLDINGCORP_04_24_2003-EX-10.13-JOINT VENTURE AGREEMENT
✅ Contract analysis stage completed


## 6. Build the Final Report and Summary Tables

This section converts the raw model outputs into a clean, presentation-friendly report.

### What this section does
This step:
- prints each full contract analysis,
- builds structured summary tables,
- displays the final outputs in a classroom-friendly format.

### Why this step matters
A good NLP demo should not end with raw API output alone.  
It should also organize the results so a human decision-maker can review them efficiently.

### Conceptual mini-flow

```text
Model Analysis Results
          ↓
Organize Into Records
          ↓
Print Human-Readable Report
          ↓
Display Summary Tables
```

In [18]:

# Define a helper function that extracts a short risk label from the model output.
def extract_risk_level(analysis_text):
    # Search for a line that begins with the risk-level heading.
    match = re.search(r"Overall risk level\s*:\s*(.+)", analysis_text, flags=re.IGNORECASE)

    # Return the captured risk label if found.
    if match:
        return match.group(1).strip()

    # Return a fallback label when the pattern is not found.
    return "Not clearly identified"

# Define a helper function that prints a readable report for all analyzed contracts.
def print_final_report(results):
    # Print a visible section header.
    print_section("STEP 6 — FINAL NLP CONTRACT ANALYSIS REPORT")

    # Loop through each analysis result one by one.
    for item in results:
        # Print the contract title.
        print(f"📄 {item['title']}")

        # Print the file name.
        print(f"File name: {item['filename']}")

        # Print the word count.
        print(f"Word count: {item['word_count']}")

        # Print the timestamp of analysis generation.
        print(f"Analysis timestamp: {item['analysis_timestamp']}")

        # Print a divider before the model output.
        print("-" * 80)

        # Print the full analysis text.
        print(item['analysis_text'])

        # Print a closing divider for the current contract.
        print("-" * 100)

# Define a helper function that builds the final summary DataFrames.
def build_final_tables(results):
    # Create an empty list for document-level summary rows.
    summary_rows = []

    # Create an empty list for full analysis rows.
    detailed_rows = []

    # Loop through each result item.
    for item in results:
        # Extract a compact risk label from the analysis text.
        risk_level = extract_risk_level(item["analysis_text"])

        # Build one summary row.
        summary_rows.append({
            "title": item["title"],
            "filename": item["filename"],
            "word_count": item["word_count"],
            "risk_level": risk_level,
            "analysis_timestamp": item["analysis_timestamp"]
        })

        # Build one detailed row.
        detailed_rows.append({
            "title": item["title"],
            "analysis_text": item["analysis_text"]
        })

    # Convert the summary rows into a DataFrame.
    summary_df = pd.DataFrame(summary_rows)

    # Convert the detailed rows into a DataFrame.
    detailed_df = pd.DataFrame(detailed_rows)

    # Return both DataFrames.
    return summary_df, detailed_df

# Print the final readable report.
print_final_report(analysis_results)


STEP 6 — FINAL NLP CONTRACT ANALYSIS REPORT
📄 2ThemartComInc_19990826_10-12G_EX-10.10_6700288_EX-10.10_Co-Branding Agreement_ Agency Agreement
File name: 2ThemartComInc_19990826_10-12G_EX-10.10_6700288_EX-10.10_Co-Branding Agreement_ Agency Agreement.txt
Word count: 4467
Analysis timestamp: 2026-03-26 07:53:36
--------------------------------------------------------------------------------
PARTIES:  
- Party A: i-Escrow, Inc. (“i-Escrow”)  
- Party B: 2TheMart.com, Inc. (“2TheMart”)  

MAIN PURPOSE:  
This Co-Branding and Advertising Agreement establishes a strategic partnership whereby 2TheMart promotes i-Escrow’s escrow services to its auction users via co-branded digital channels, and i-Escrow implements and operates those services—including a custom-integrated, co-branded website (www.iescrow.com/2TheMart) and an automated information transfer mechanism—to enable seamless transaction initiation. The agreement governs joint branding, content licensing, advertising compensation, ser

In [19]:
# Build the final DataFrames used for notebook display.
summary_df, detailed_df = build_final_tables(analysis_results)

In [20]:
# Print a visible title before displaying the compact summary table.
print_section("DOCUMENT-LEVEL SUMMARY TABLE")

# Display the compact summary table.
display(summary_df)


DOCUMENT-LEVEL SUMMARY TABLE


,title,filename,word_count,risk_level,analysis_timestamp
0,2ThemartComInc_19990826_10-12G_EX-10.10_670028...,2ThemartComInc_19990826_10-12G_EX-10.10_670028...,4467,**Medium**,2026-03-26 07:53:36
1,ABILITYINC_06_15_2020-EX-4.25-SERVICES AGREEMENT,ABILITYINC_06_15_2020-EX-4.25-SERVICES AGREEME...,4115,**Medium–High**,2026-03-26 07:54:32
2,ACCELERATEDTECHNOLOGIESHOLDINGCORP_04_24_2003-...,ACCELERATEDTECHNOLOGIESHOLDINGCORP_04_24_2003-...,1911,Medium,2026-03-26 07:55:28


In [21]:
# Print a visible title before displaying the full analysis table.
print_section("FULL CONTRACT ANALYSIS TABLE")

# Display the full analysis table.
display(detailed_df)


FULL CONTRACT ANALYSIS TABLE


,title,analysis_text
0,2ThemartComInc_19990826_10-12G_EX-10.10_670028...,"PARTIES: \n- Party A: i-Escrow, Inc. (“i-Escr..."
1,ABILITYINC_06_15_2020-EX-4.25-SERVICES AGREEMENT,PARTIES: \n- Party A: [ * * * ] (the “Provide...
2,ACCELERATEDTECHNOLOGIESHOLDINGCORP_04_24_2003-...,PARTIES: \n- Party A: Collectible Concepts Gr...
